# URECA: DEUDF reruns (aligned with the official code) + NSH re-extraction

**Runtime -> Change runtime type -> T4 GPU** before you do anything else.

This notebook does two things:

| step | what | cost |
|---|---|---|
| NSH | re-extract and re-evaluate the finished 10K-iteration runs from their final checkpoints in Drive, with the new component filter (keep every component ≥ 5% of the largest) | minutes, no retraining |
| DEUDF | retrain on torus, thin_plate and cube_sheet with the aligned method: 30K iterations, res 256, data-anchored iso-level | ~30-60 min per shape |

MPF is not run here; it comes back once Stage 2 (MPF debugging) is done.

Outputs go to Google Drive under `RUN_DIR/<variant>/<shape>_<method>/`: `metrics.json`,
`loss.png`, `recon.ply`, `input.ply`, `train.log` and `ckpt.pt`. NSH runs are in
`main/` (re-extraction keeps the originals as `metrics_orig.json` / `recon_orig.ply`);
the new DEUDF runs go to `deudf_v2/`, so the earlier DEUDF runs in `main/` are not
touched and not skipped over.

**If Colab disconnects:** rerun cells 1-4, then the remaining run cells. Finished runs
are skipped; an interrupted run resumes from its last checkpoint (every 1000
iterations). `force=True` restarts a run from scratch.

In [ ]:
# 1. clone the fix-bugs branch and install requirements
import os, subprocess
REPO, BRANCH = "/content/ureca3d", "fix-bugs"
if not os.path.isdir(REPO):
    !git clone -q -b {BRANCH} https://github.com/bhavyapuri15/ureca3d.git {REPO}
else:
    !git -C {REPO} fetch -q origin && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} pull -q --ff-only
%cd {REPO}
!pip -q install -r requirements.txt fast_simplification   # the latter only for the gallery
COMMIT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True).strip()
print("branch", BRANCH, "at", COMMIT)

In [ ]:
# 2. check the GPU is really there -- 10K iterations on CPU would take days
import torch
print(torch.__version__)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# 3. mount Google Drive -- every run's outputs go here
from google.colab import drive
drive.mount("/content/drive")
RUN_NAME = "fix-bugs"          # change this to start a fresh set of runs
RUN_DIR = f"/content/drive/MyDrive/ureca3d_runs/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)
print("outputs ->", RUN_DIR)

In [ ]:
# 4. run helper: streams the training log to this notebook AND to Drive,
#    skips runs that already finished, resumes interrupted ones from ckpt.pt
import json, sys, shlex

def run(method, shape, res=256, extra=(), variant="main", iters=10000,
        log_every=None, ckpt_every=1000, force=False):
    out_root = os.path.join(RUN_DIR, variant)
    odir = os.path.join(out_root, f"{shape}_{method}")
    metrics, ckpt = os.path.join(odir, "metrics.json"), os.path.join(odir, "ckpt.pt")
    if os.path.exists(metrics) and not force:
        print(f"[skip] {variant}/{shape}_{method} already finished -- force=True to redo")
        return
    if force:
        for f in (metrics, ckpt):
            if os.path.exists(f): os.remove(f)
    os.makedirs(odir, exist_ok=True)
    if log_every is None:
        log_every = 100 if method == "mpf" else 250   # finer eik_r trace for MPF
    cmd = [sys.executable, "-u", "train.py", "--method", method, "--shape", shape,
           "--iters", str(iters), "--res", str(res), "--out", out_root,
           "--log_every", str(log_every), "--ckpt_every", str(ckpt_every), "--resume"]
    cmd += list(extra)
    log_path = os.path.join(odir, "train.log")
    # keep the earlier part of the log when resuming
    with open(log_path, "a" if os.path.exists(ckpt) else "w") as log:
        log.write(f"# commit {COMMIT}\n# {shlex.join(cmd)}\n"); log.flush()
        p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="")
            log.write(line); log.flush()
        rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"{variant}/{shape}_{method} exited with code {rc} -- see {log_path}")

In [ ]:
# 5. property tests -- run these before any long training job
!python -m pytest tests -q

## Re-score finished runs from their checkpoints (no retraining)

Re-extracts every finished NSH run (`main/`) and DEUDF run (`deudf_v2/`) at res 256
with the current code: component filter (keep every component ≥ 5% of the largest)
and metrics in the ground truth's frame. Runs already scored this way are skipped
(`FORCE = True` to redo). The first originals are kept as `metrics_orig.json` /
`recon_orig.ply`, so the summary below can show old and new numbers side by side.

In [ ]:
# 6. re-score finished NSH + DEUDF runs from their final checkpoints
FORCE = False
for variant, method in (("main", "nsh"), ("deudf_v2", "deudf")):
    for shape in ("torus", "thin_plate", "cube_sheet"):
        d = os.path.join(RUN_DIR, variant, f"{shape}_{method}")
        if not os.path.exists(os.path.join(d, "ckpt.pt")):
            print(f"[missing] {d}/ckpt.pt -- skipped"); continue
        mp = os.path.join(d, "metrics.json")
        if (not FORCE and os.path.exists(mp)
                and json.load(open(mp)).get("metric_frame") == "gt"):
            print(f"[done] {variant}/{shape}_{method} already scored in the GT frame"); continue
        !cd {REPO} && python reextract.py "{d}" --res 256

## DEUDF: 30K iterations, res 256

One run per cell. Watch `dist` in the log: it should fall towards 0 (it plateaued
at ~0.02 before the fix). After training, the `[extract] data level` line reports
f at the input points, which should be ≈ 0.

In [ ]:
run("deudf", "torus", res=256, iters=30000, variant="deudf_v2")

In [ ]:
run("deudf", "thin_plate", res=256, iters=30000, variant="deudf_v2")

In [ ]:
run("deudf", "cube_sheet", res=256, iters=30000, variant="deudf_v2")

## Results

In [ ]:
# 7. summary: old vs new numbers side by side (also written to RUN_DIR/summary.md)
#    reported  = metrics as first written (metrics_orig.json): old filter, old metric
#    old metric = current mesh, each mesh normalised by its own bbox
#    new        = current mesh, both normalised with the ground truth's transform
EXPECTED = ([("main", "nsh", s) for s in ("torus", "thin_plate", "cube_sheet")]
            + [("deudf_v2", "deudf", s) for s in ("torus", "thin_plate", "cube_sheet")])

def fmt(v, nd=2):
    if isinstance(v, bool) or isinstance(v, int): return str(v)
    return f"{v:.{nd}f}" if isinstance(v, float) else ("—" if v is None else str(v))

head = ["method", "shape", "F reported", "F old metric", "F new ↑", "CD reported",
        "CD new ↓", "NC new ↑", "#comp (kept)", "f@data median / max", "iters"]
lines = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
for variant, method, shape in EXPECTED:
    d = os.path.join(RUN_DIR, variant, f"{shape}_{method}")
    mp, op = os.path.join(d, "metrics.json"), os.path.join(d, "metrics_orig.json")
    if not os.path.exists(mp):
        lines.append(f"| {method} | {shape} | *not run / unfinished* |" + " |" * (len(head) - 3))
        continue
    m = json.load(open(mp))
    o = json.load(open(op)) if os.path.exists(op) else {}
    new = m.get("metric_frame") == "gt"
    fdata = (f"{m['f_data_median']:.4f} / {m['f_data_max']:.4f}" if "f_data_median" in m else None)
    lines.append("| " + " | ".join([
        method, shape,
        fmt(o.get("f_score", m.get("f_score_each_bbox"))),
        fmt(m.get("f_score_each_bbox")),
        fmt(m.get("f_score")) if new else "*not re-scored*",
        fmt(o.get("chamfer_l1_x1e3", m.get("chamfer_l1_x1e3_each_bbox"))),
        fmt(m.get("chamfer_l1_x1e3")) if new else "—",
        fmt(m.get("normal_consistency")),
        f"{m.get('n_components')} ({m.get('n_kept', '—')})", fmt(fdata), fmt(m.get("iters")),
    ]) + " |")

table = "\n".join(lines)
print(f"commit {COMMIT}\n")
print(table)
open(os.path.join(RUN_DIR, "summary.md"), "w").write(f"commit {COMMIT}\n\n{table}\n")

In [ ]:
# 8. render every reconstruction to a PNG grid (RUN_DIR/gallery.png)
import numpy as np, trimesh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def show(ax, path, title):
    ax.set_title(title, fontsize=8); ax.set_axis_off()
    if not os.path.exists(path): ax.text(0, 0, 0, 'missing'); return
    m = trimesh.load(path, process=False)
    faces = m.faces
    if len(faces) > 60000:
        try:
            m = m.simplify_quadric_decimation(face_count=60000); faces = m.faces
        except Exception:   # no fast_simplification: draw a random subset instead
            faces = faces[np.random.default_rng(0).choice(len(faces), 60000, replace=False)]
    tri = m.vertices[faces]
    pc = Poly3DCollection(tri, facecolor='#c9ced6', edgecolor='none', alpha=1.0)
    ax.add_collection3d(pc)
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
    ax.view_init(elev=22, azim=38)

n = len(EXPECTED); cols = 3; rows_ = (n + cols - 1) // cols
fig = plt.figure(figsize=(3.1 * cols, 3.1 * rows_))
for i, (variant, method, shape) in enumerate(EXPECTED):
    ax = fig.add_subplot(rows_, cols, i + 1, projection='3d')
    show(ax, os.path.join(RUN_DIR, variant, f"{shape}_{method}", "recon.ply"),
         f"{shape}_{method}")
fig.tight_layout(); fig.savefig(os.path.join(RUN_DIR, 'gallery.png'), dpi=150)
print('written', os.path.join(RUN_DIR, 'gallery.png'))